# Full fine-tune Qwen3.5-4B for the Decillion dsh agentic system (Kaggle, 2×T4)

This version replaces Unsloth/QLoRA with **full-parameter fine-tuning** using TRL + DeepSpeed ZeRO-3 CPU offload.

Run cells in order. It expects a Kaggle notebook with two Tesla T4 GPUs and internet access enabled.

1. Configure CUDA allocator and inspect both GPUs
2. Set Kaggle-local output paths
3. Install Transformers / TRL / DeepSpeed (without explicitly reinstalling PyTorch)
4. Configure model, data mix and conservative T4 sequence length
5. Build the dataset and choose the training mix
6. Load the Qwen processor/tokenizer and render the existing tool-aware chat rows
7. Write the ZeRO-3 configuration and a standalone distributed training script
8. Launch two workers with `torchrun`
9. Read evaluation metrics and verify the final merged model
10. Serve with vLLM, expose with Cloudflare, and run the existing API tests

**Important:** Kaggle's `/kaggle/working` storage is temporary. Download the final model/checkpoints from the notebook Output panel or upload them separately before the session ends. This notebook does not pretend that a PyDrive2 API client mounts Google Drive as a filesystem.

The training script preserves assistant-only loss: it trains on assistant messages (including assistant tool calls) and masks system, user, and tool-result messages. The model is loaded with the official Qwen3.5 multimodal AutoModel class, but this dataset is text-only; visual inputs are not trained.


In [1]:
import os

# Set before importing torch. Helps allocator segment growth; it does not create VRAM.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [2]:
# 1. Kaggle GPU and host-RAM check
import torch
import psutil

assert torch.cuda.is_available(), "No CUDA GPU: enable GPU in Kaggle Notebook settings"
NUM_GPUS = torch.cuda.device_count()
print("Visible GPUs:", NUM_GPUS)
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {props.name} | total={total / 1024**3:.2f} GiB | "
        f"free={free / 1024**3:.2f} GiB"
    )

ram = psutil.virtual_memory()
print(f"RAM: total={ram.total / 1024**3:.2f} GiB | available={ram.available / 1024**3:.2f} GiB")
if NUM_GPUS != 2:
    raise RuntimeError(f"Expected 2 GPUs, but found {NUM_GPUS}. Select Kaggle's 2×T4 accelerator.")


Visible GPUs: 2
GPU 0: Tesla T4 | total=14.56 GiB | free=14.46 GiB
GPU 1: Tesla T4 | total=14.56 GiB | free=14.46 GiB
RAM: total=31.35 GiB | available=29.41 GiB


In [3]:
# 2. Kaggle-local output directory
# Kaggle's /kaggle/working is persisted as notebook output when you Save Version.
import os

OUT_DIR = "/kaggle/working/decillion-qwen35-4b"
os.makedirs(OUT_DIR, exist_ok=True)
print("outputs →", OUT_DIR)
print("Note: copy/download important checkpoints and the final model before the Kaggle session expires.")


outputs → /kaggle/working/decillion-qwen35-4b
Note: copy/download important checkpoints and the final model before the Kaggle session expires.


In [5]:
# 3. Install the standard Transformers + TRL + DeepSpeed stack.
# Do not explicitly reinstall torch: keep Kaggle's CUDA-enabled PyTorch build.
!pip install -q -U transformers trl accelerate deepspeed datasets psutil jsonschema openai requests

# Restart the Kaggle session if pip reports that core packages (especially torch) were changed.


In [6]:
# 4. Configuration
MODEL_ID = "Qwen/Qwen3.5-4B"
REPO = "https://github.com/keyhmoham/Aseman.git"
BRANCH = "main"
DATA_DIR = "/kaggle/working/Aseman/datasets/decillion-dsh-agentic"

# Conservative first run for 2×T4 full fine-tuning. Increase only after a successful smoke test.
MAX_SEQ_LEN = 512

# Full fine-tuning (no LoRA, no 4-bit quantization)
EPOCHS = 1
LEARNING_RATE = 2e-5
BATCH_SIZE = 1
GRAD_ACCUM = 8
WARMUP_RATIO = 0.03
SAVE_STEPS = 100

# Data mix: multi-turn sessions are ~94% of the data; single-turn families are repeated to balance it.
SINGLE_TURN_REPEAT = 4
MAX_MULTI_TURN = 30000  # cap on session.* rows per epoch (None = all)
TOOLS_MODE = "relevant"  # 'relevant' or 'full'
SEED = 3407

DS_CONFIG = "/kaggle/working/ds_zero3_cpu.json"
TRAIN_SCRIPT = "/kaggle/working/train_deepspeed.py"
TRAIN_DATA_DIR = "/kaggle/working/train_dataset"
VAL_DATA_DIR = "/kaggle/working/val_dataset"
MERGED_DIR = f"{OUT_DIR}/merged"

print(dict(
    MODEL_ID=MODEL_ID, MAX_SEQ_LEN=MAX_SEQ_LEN, EPOCHS=EPOCHS,
    BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM, NUM_GPUS=NUM_GPUS,
    mode="full fine-tuning + ZeRO-3 CPU offload"
))


{'MODEL_ID': 'Qwen/Qwen3.5-4B', 'MAX_SEQ_LEN': 512, 'EPOCHS': 1, 'BATCH_SIZE': 1, 'GRAD_ACCUM': 8, 'NUM_GPUS': 2, 'mode': 'full fine-tuning + ZeRO-3 CPU offload'}


In [7]:
# 5. Get the dataset from GitHub and build the training rows
import os

REPO_DIR = "/kaggle/working/Aseman"
if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

!python3 {DATA_DIR}/scripts/build.py --tools {TOOLS_MODE} --out /kaggle/working/dist
!ls -la /kaggle/working/dist
!cat /kaggle/working/dist/stats.json | head -40


{"train": 66918, "validation": 1421}
total 977380
drwxr-xr-x 2 root root      4096 Nov 22 05:01 .
drwxr-xr-x 7 root root      4096 Nov 22 04:57 ..
-rw-r--r-- 1 root root      2670 Nov 22 05:01 stats.json
-rw-r--r-- 1 root root 980101154 Nov 22 05:01 train.jsonl.gz
-rw-r--r-- 1 root root  20717530 Nov 22 05:01 validation.jsonl.gz
{
 "train": {
  "total": 66918,
  "by_category": {
   "api.confined": 1,
   "api.policy_denied": 2,
   "api.search_call": 35,
   "apps.file_intent_denied": 1,
   "apps.function": 58,
   "apps.image_generate": 6,
   "apps.mcp_add": 5,
   "apps.plain_words": 12,
   "apps.routine_daily": 32,
   "apps.untrusted_output": 2,
   "appui.build_publish": 15,
   "appui.offer_first": 1,
   "dev.bugfix": 513,
   "dev.edit_replace_all": 17,
   "dev.read_long_file": 4,
   "dev.script_deliver": 29,
   "memory.disabled": 1,
   "memory.recall": 96,
   "memory.recall_miss": 2,
   "memory.remember": 118,
   "memory.rewrite_notes": 1,
   "memory.session_search": 2,
   "memory.timel

In [8]:
# 6. Choose the training mix (rows stay raw JSON lines to save memory)
import gzip, json, random, re
from collections import Counter

random.seed(SEED)
CAT = re.compile(r'"category": "([^"]+)"')
single, multi = [], []
with gzip.open('/kaggle/working/dist/train.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        match = CAT.search(line[:300])
        if not match:
            continue
        cat = match.group(1)
        (multi if cat.startswith('session.') else single).append(line)

random.shuffle(multi)
if MAX_MULTI_TURN:
    multi = multi[:MAX_MULTI_TURN]
train_lines = single * SINGLE_TURN_REPEAT + multi
random.shuffle(train_lines)
with gzip.open('/kaggle/working/dist/validation.jsonl.gz', 'rt', encoding='utf-8') as f:
    val_lines = f.read().splitlines()

print(f'single-turn {len(single)} x{SINGLE_TURN_REPEAT}, multi-turn {len(multi)} -> train {len(train_lines)}, validation {len(val_lines)}')


single-turn 2280 x4, multi-turn 30000 -> train 39120, validation 1421


In [9]:
# 7. Load Qwen3.5 processor/tokenizer. The model itself is loaded in each torchrun worker.
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tok = tokenizer
print("Processor:", type(processor).__name__)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab size:", len(tokenizer))


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Processor: Qwen3VLProcessor
Tokenizer: Qwen3_5Tokenizer
Vocab size: 248077


In [15]:
# 8. Render and filter datasets without Dataset.from_generator

import json
import shutil
from tqdm.auto import tqdm
from datasets import Dataset, Features, Value

FEATURES = Features({
    "text": Value("string")
})

def prepare_rows(lines, split_name):
    prepared = []
    rejected_long = 0
    rejected_errors = 0

    for line in tqdm(lines, desc=f"Preparing {split_name}"):
        try:
            row = json.loads(line) if isinstance(line, str) else line

            text = processor.apply_chat_template(
                row["messages"],
                tools=row.get("tools"),
                tokenize=False,
            )

            token_ids = tokenizer(
                text,
                add_special_tokens=False,
                truncation=False,
            )["input_ids"]

            if len(token_ids) <= MAX_SEQ_LEN:
                prepared.append({"text": text})
            else:
                rejected_long += 1

        except Exception as e:
            rejected_errors += 1
            if rejected_errors <= 5:
                print(f"\nError processing {split_name}: {e}")

    print(f"\n{split_name} statistics:")
    print(f"Original rows: {len(lines)}")
    print(f"Retained rows: {len(prepared)}")
    print(f"Too long: {rejected_long}")
    print(f"Errors: {rejected_errors}")

    return prepared

# Process the data
train_rows = prepare_rows(train_lines, "train")
val_rows = prepare_rows(val_lines, "validation")

# Explicit features also support empty lists.
train_ds = Dataset.from_list(train_rows, features=FEATURES)
val_ds = Dataset.from_list(val_rows, features=FEATURES)

if len(train_ds) == 0:
    raise RuntimeError(
        "Training dataset is empty. Check the dataset source, "
        "chat template, and MAX_SEQ_LEN."
    )

if len(val_ds) == 0:
    raise RuntimeError(
        "Validation dataset is empty. Check the dataset source, "
        "chat template, and MAX_SEQ_LEN."
    )

print("\nSample:")
print(train_ds[0]["text"][-1500:])

# Persist the datasets
for path in (TRAIN_DATA_DIR, VAL_DATA_DIR):
    shutil.rmtree(path, ignore_errors=True)

train_ds.save_to_disk(TRAIN_DATA_DIR)
val_ds.save_to_disk(VAL_DATA_DIR)

print("\nDatasets saved successfully.")
print("Training:", TRAIN_DATA_DIR)
print("Validation:", VAL_DATA_DIR)

Preparing train:   0%|          | 0/39120 [00:00<?, ?it/s]


train statistics:
Original rows: 39120
Retained rows: 0
Too long: 39120
Errors: 0


Preparing validation:   0%|          | 0/1421 [00:00<?, ?it/s]


validation statistics:
Original rows: 1421
Retained rows: 0
Too long: 1421
Errors: 0


ValueError: Keys mismatch: between {'text': Value('string')} (source) and {} (target).
{'text'} are missing from target and set() are missing from source
The 'source' features come from dataset_info.json, and the 'target' ones are those of the dataset arrow file.

In [ ]:
# 9. Write DeepSpeed ZeRO-3 CPU-offload config and the standalone TRL training script
import json

# Both optimizer state and inactive parameter shards can live in host RAM.
# This trades GPU VRAM for RAM and CPU↔GPU transfer time.
ds_config = {
    "fp16": {"enabled": "auto"},
    "bf16": {"enabled": False},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": False},
        "offload_param": {"device": "cpu", "pin_memory": False},
        "overlap_comm": False,
        "contiguous_gradients": True,
        "reduce_bucket_size": 50000000,
        "stage3_prefetch_bucket_size": 50000000,
        "stage3_param_persistence_threshold": 100000,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "zero_force_ds_cpu_optimizer": False,
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto"
}
with open(DS_CONFIG, "w") as f:
    json.dump(ds_config, f, indent=2)
print("Wrote", DS_CONFIG)

train_script = r"""import os, glob, json, math
import torch
from datasets import load_from_disk
from transformers import AutoProcessor, AutoModelForMultimodalLM
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen3.5-4B"
OUT_DIR = "/kaggle/working/decillion-qwen35-4b"
DS_CONFIG = "/kaggle/working/ds_zero3_cpu.json"
TRAIN_DATA_DIR = "/kaggle/working/train_dataset"
VAL_DATA_DIR = "/kaggle/working/val_dataset"
MAX_SEQ_LEN = 512
EPOCHS = 1
LEARNING_RATE = 2e-5
BATCH_SIZE = 1
GRAD_ACCUM = 8
SAVE_STEPS = 100
SEED = 3407
CHECKPOINT_DIR = f"{OUT_DIR}/checkpoints"
FINAL_DIR = f"{OUT_DIR}/merged"
os.makedirs(OUT_DIR, exist_ok=True)

# IMPORTANT: create SFTConfig/TrainingArguments BEFORE loading the model.
# Transformers uses this to activate ZeRO-3 parameter partitioning during from_pretrained.
args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    dataset_kwargs={"skip_prepare_dataset": True},
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    deepspeed=DS_CONFIG,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
    max_grad_norm=1.0,
)

# Importing/constructing TrainingArguments above initializes the HF DeepSpeed integration.
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the official multimodal architecture. The current training dataset is text-only.
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
model.config.use_cache = False

train_ds = load_from_disk(TRAIN_DATA_DIR)
val_ds = load_from_disk(VAL_DATA_DIR)

class AssistantOnlyCollator:
    # Tokenize rendered Qwen chat text; compute loss only on assistant message bodies.
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
        self.im_sep = tokenizer.convert_tokens_to_ids("<|im_sep|>")
        if self.im_start == tokenizer.unk_token_id or self.im_sep == tokenizer.unk_token_id:
            raise ValueError("Tokenizer is missing Qwen chat boundary tokens <|im_start|>/<|im_sep|>")

    def __call__(self, features):
        texts = [x["text"] for x in features]
        batch = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=False,
        )
        ids = batch["input_ids"]
        masks = batch["attention_mask"]
        labels = torch.full_like(ids, -100)

        for b in range(ids.shape[0]):
            valid = int(masks[b].sum().item())
            seq = ids[b, :valid].tolist()
            starts = [i for i, token in enumerate(seq) if token == self.im_start]
            for j, start in enumerate(starts):
                next_start = starts[j + 1] if j + 1 < len(starts) else valid
                try:
                    sep = seq.index(self.im_sep, start + 1, next_start)
                except ValueError:
                    continue
                header = self.tokenizer.decode(
                    seq[start + 1:sep], skip_special_tokens=False
                ).strip()
                # Covers ordinary assistant turns and assistant tool-call headers (assistant to=...).
                if header.startswith("assistant"):
                    body_start = sep + 1
                    labels[b, body_start:next_start] = ids[b, body_start:next_start]

        batch["labels"] = labels
        return batch

collator = AssistantOnlyCollator(tokenizer, MAX_SEQ_LEN)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds.select(range(min(300, len(val_ds)))),
    processing_class=tokenizer,
    data_collator=collator,
)

# Resume from the latest local checkpoint if this Kaggle session still has it.
last = sorted(
    glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"),
    key=lambda p: int(p.rsplit("-", 1)[1])
)
resume = last[-1] if last else None
print("Resuming from:", resume)
print("Train examples:", len(train_ds), "| validation examples:", len(val_ds))
print("World size:", os.environ.get("WORLD_SIZE", "unknown"))

train_result = trainer.train(resume_from_checkpoint=resume)

# Evaluate on all ranks so distributed collectives remain synchronized.
metrics = trainer.evaluate(eval_dataset=val_ds.select(range(min(300, len(val_ds)))))
if "eval_loss" in metrics:
    metrics["eval_perplexity"] = math.exp(min(float(metrics["eval_loss"]), 20))
if trainer.is_world_process_zero():
    with open(f"{OUT_DIR}/eval_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

# With stage3_gather_16bit_weights_on_model_save=true, save_model gathers the
# ZeRO-sharded 16-bit weights into a standard Hugging Face model directory.
trainer.save_model(FINAL_DIR)
if trainer.is_world_process_zero():
    processor.save_pretrained(FINAL_DIR)
    print("Final model saved to:", FINAL_DIR)
    print("Evaluation:", metrics)
"""

with open(TRAIN_SCRIPT, "w") as f:
    f.write(train_script)
print("Wrote training script:", TRAIN_SCRIPT)


In [ ]:
# 10. Launch full fine-tuning on both T4 GPUs
# This starts one worker per GPU. Do not create a Trainer in the notebook kernel.
!CUDA_VISIBLE_DEVICES=0,1 torchrun --standalone --nnodes=1 --nproc_per_node=2 --master_port=29501 {TRAIN_SCRIPT}


In [ ]:
# 11. Read held-out validation metrics
import json, os

metrics_path = f"{OUT_DIR}/eval_metrics.json"
if os.path.exists(metrics_path):
    metrics = json.load(open(metrics_path))
    print(json.dumps(metrics, indent=2))
else:
    print("No evaluation metrics file found. Check the torchrun output for a training/evaluation error.")


In [ ]:
# 12. Verify final 16-bit Hugging Face model and release notebook-process GPU memory
import os, gc, torch

MERGED_DIR = f"{OUT_DIR}/merged"
assert os.path.exists(f"{MERGED_DIR}/config.json"), (
    f"No consolidated model found at {MERGED_DIR}. Check the torchrun logs and DeepSpeed save step."
)
print("Final model directory:", MERGED_DIR)
print("Files:", os.listdir(MERGED_DIR)[:20])
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 13. Serve the merged model with vLLM: an OpenAI-compatible API on port 8000
# Run after training in the same Kaggle session; model is under /kaggle/working.
import json, os, shutil, subprocess, sys, time, urllib.request
from importlib.metadata import PackageNotFoundError, version
import torch

OUT_DIR = globals().get('OUT_DIR', '/kaggle/working/decillion-qwen35-4b')
MODEL_ID = globals().get('MODEL_ID', 'Qwen/Qwen3.5-4B')
MERGED_DIR = globals().get('MERGED_DIR', f'{OUT_DIR}/merged')
SERVED_NAME = 'decillion-dsh-qwen35-4b'
PORT = 8000
MAX_LEN = 4096  # conservative inference context for 2×T4; lower if vLLM runs out of VRAM
VLLM_LOG = '/kaggle/working/vllm.log'

assert os.path.exists(f'{MERGED_DIR}/config.json'), f'no merged model at {MERGED_DIR}: run cell 12 first'

# The merged folder can miss the processor/chat-template files of the multimodal base model.
from huggingface_hub import snapshot_download
base = snapshot_download(MODEL_ID, allow_patterns=[
    'preprocessor_config.json', 'video_preprocessor_config.json', 'processor_config.json',
    'chat_template.jinja', 'generation_config.json'])
for name in os.listdir(base):
    if os.path.isfile(f'{base}/{name}') and not os.path.exists(f'{MERGED_DIR}/{name}'):
        shutil.copy2(f'{base}/{name}', f'{MERGED_DIR}/{name}')

# Serve from local disk: mmap over the Drive FUSE mount is slow and unreliable.
LOCAL_DIR = '/kaggle/working/merged'
stamp = f'{LOCAL_DIR}/.source-mtime'
src_mtime = os.path.getmtime(f'{MERGED_DIR}/config.json')
stale = (not os.path.exists(f'{LOCAL_DIR}/config.json') or not os.path.exists(stamp)
         or float(open(stamp).read()) < src_mtime)
if stale:
    print('copying the merged model to local disk (a few minutes)…')
    shutil.rmtree(LOCAL_DIR, ignore_errors=True)
    shutil.copytree(MERGED_DIR, LOCAL_DIR)
    open(stamp, 'w').write(str(src_mtime))

try:
    have_vllm = tuple(int(x) for x in version('vllm').split('.')[:2]) >= (0, 34)
except (PackageNotFoundError, ValueError):
    have_vllm = False
if not have_vllm:
    print('installing vLLM (a few minutes; it brings its own torch)…')
    !pip install -q -U 'vllm>=0.34.0'

# The tunnel publishes this port to the internet, so the API always needs a key.
API_KEY = globals().get('API_KEY') or 'sk-decillion-' + os.urandom(16).hex()
# T4 does not support bf16.
DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'

subprocess.run(['pkill', '-f', 'vllm.entrypoints.openai.api_server'], capture_output=True)
time.sleep(3)
cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', LOCAL_DIR,
    '--served-model-name', SERVED_NAME,
    '--host', '127.0.0.1', '--port', str(PORT),
    '--api-key', API_KEY,
    '--dtype', DTYPE,
    '--max-model-len', str(MAX_LEN),
    '--tensor-parallel-size', '2',
    '--gpu-memory-utilization', '0.9' if DTYPE == 'float16' else '0.85',
    '--max-num-seqs', '2',
    # dsh's agentic calls: Qwen's XML <tool_call> blocks and its  thinking block
    '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_xml', '--reasoning-parser', 'qwen3',
    # this fine-tune is text-only: do not reserve memory for images or video
    '--limit-mm-per-prompt', '{"image": 0, "video": 0}',
]
if DTYPE == 'float16':
    cmd += ['--enforce-eager']   # T4: skip CUDA graphs to reduce memory pressure
print('starting vLLM…')
proc = subprocess.Popen(cmd, stdout=open(VLLM_LOG, 'w'), stderr=subprocess.STDOUT)

def served_ids():
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/models',
                                 headers={'Authorization': f'Bearer {API_KEY}'})
    return [m['id'] for m in json.load(urllib.request.urlopen(req, timeout=5))['data']]

deadline, ids = time.time() + 900, None
while time.time() < deadline and proc.poll() is None:
    try:
        ids = served_ids()
        break
    except Exception:
        time.sleep(5)
if ids != [SERVED_NAME]:
    print(open(VLLM_LOG).read()[-4000:])
    raise RuntimeError('vLLM did not come up: see its log above')

SERVE = {'base_url_local': f'http://127.0.0.1:{PORT}/v1', 'port': PORT, 'model': SERVED_NAME,
         'api_key': API_KEY, 'max_model_len': MAX_LEN, 'merged_dir': MERGED_DIR}
json.dump(SERVE, open(f'{OUT_DIR}/serve.json', 'w'), indent=1)
print('serving', SERVED_NAME, '→', SERVE['base_url_local'])
print('API key:', API_KEY, '| saved to', f'{OUT_DIR}/serve.json')

In [ ]:
# 14. Expose the vLLM API to the internet with a Cloudflare quick tunnel (no account needed)
import json, os, re, subprocess, time, urllib.request

OUT_DIR = globals().get('OUT_DIR', '/kaggle/working/decillion-qwen35-4b')
SERVE = json.load(open(f'{OUT_DIR}/serve.json'))
assert 'base_url_local' in SERVE, 'run cell 13 first'
PORT, API_KEY = SERVE['port'], SERVE['api_key']
CF, CF_LOG = '/usr/local/bin/cloudflared', '/kaggle/working/cloudflared.log'

if not os.path.exists(CF):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF}
    !chmod +x {CF}

# Make sure the model is still up before publishing it.
req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/models',
                             headers={'Authorization': f'Bearer {API_KEY}'})
urllib.request.urlopen(req, timeout=10).read()

subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(2)
if os.path.exists(CF_LOG):
    os.remove(CF_LOG)
# HTTP/2 uses TCP 443 and is often more reliable in hosted notebook environments.
subprocess.Popen([CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate',
                  '--protocol', 'http2', '--logfile', CF_LOG],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

url, deadline = None, time.time() + 90
while time.time() < deadline and not url:
    time.sleep(2)
    if os.path.exists(CF_LOG):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(CF_LOG).read())
        url = m.group(0) if m else None
if not url:
    print(open(CF_LOG).read()[-2000:] if os.path.exists(CF_LOG) else '')
    raise RuntimeError('cloudflared printed no URL')
print('public URL:', url)

def public_ids():
    req = urllib.request.Request(f'{url}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'})
    return [m['id'] for m in json.load(urllib.request.urlopen(req, timeout=10))['data']]

deadline = time.time() + 120
while True:
    try:
        ids = public_ids()
        break
    except Exception as exc:
        if time.time() > deadline:
            raise RuntimeError(f'the tunnel is up but the API is not reachable yet: {exc}')
        time.sleep(3)

SERVE['base_url_public'] = url + '/v1'
json.dump(SERVE, open(f'{OUT_DIR}/serve.json', 'w'), indent=1)
print('reachable from the internet →', SERVE['base_url_public'], '| model', ids[0])
print('This URL lives only while this runtime and cloudflared stay up; rerun this cell and update')
print('the provider in Decillion if it changes. A named Cloudflare tunnel (dashboard → token) gives')
print('a stable hostname instead: cloudflared tunnel run --token <TOKEN>.')

In [ ]:
# 15. Test the public API end to end, the way the Decillion LLM gateway uses it
import gzip, json, os, time
import requests
from jsonschema import validate
from openai import OpenAI

OUT_DIR = globals().get('OUT_DIR', '/kaggle/working/decillion-qwen35-4b')
DATA_DIR = globals().get('DATA_DIR', '/kaggle/working/Aseman/datasets/decillion-dsh-agentic')
SERVE = json.load(open(f'{OUT_DIR}/serve.json'))
assert 'base_url_public' in SERVE, 'run cell 14 first: the public URL is missing from serve.json'
BASE, KEY, MODEL = SERVE['base_url_public'].rstrip('/'), SERVE['api_key'], SERVE['model']
client = OpenAI(base_url=BASE, api_key=KEY, timeout=300, max_retries=1)
HEADERS = {'Authorization': f'Bearer {KEY}'}

results = []
def check(name, ok, detail=''):
    results.append((name, bool(ok)))
    print(('PASS' if ok else 'FAIL'), '|', name, ('— ' + detail if detail else ''))

# 1. The OpenAI surface the gateway expects, with and without the key.
r = requests.get(f'{BASE}/models', timeout=30)
check('refuses requests without the key', r.status_code == 401, f'HTTP {r.status_code}')
r = requests.get(f'{BASE}/models', headers=HEADERS, timeout=30)
ids = [m.get('id') for m in r.json().get('data', [])] if r.status_code == 200 else []
check('/v1/models lists the fine-tuned model', ids == [MODEL], str(ids))

# 2. A plain completion.
resp = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with exactly: DECILLION-OK'}],
    temperature=0, max_tokens=64, extra_body={'chat_template_kwargs': {'enable_thinking': False}})
text = (resp.choices[0].message.content or '').strip()
check('a chat completion comes back', 'DECILLION-OK' in text, repr(text[:80]))

# 3. The gateway's own probe shape: streaming, with usage, timed.
t0 = time.time()
first = usage = None
streamed = reasoned = ''
body = {'model': MODEL, 'stream': True, 'stream_options': {'include_usage': True},
        'messages': [{'role': 'user', 'content': 'In about 60 words, explain why the sky is blue.'}],
        'max_tokens': 300}
with requests.post(f'{BASE}/chat/completions', json=body, stream=True, timeout=300,
                   headers={**HEADERS, 'Accept': 'text/event-stream'}) as r:
    r.raise_for_status()
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data:'):
            continue
        payload = line[5:].strip()
        if payload == '[DONE]':
            break
        chunk = json.loads(payload)
        if chunk.get('usage'):
            usage = chunk['usage']
        for choice in chunk.get('choices', []):
            delta = choice.get('delta') or {}
            if first is None and (delta.get('content') or delta.get('reasoning_content') or delta.get('tool_calls')):
                first = time.time() - t0
            streamed += delta.get('content') or ''
            reasoned += delta.get('reasoning_content') or ''
check('streams tokens and reports usage for metering',
      first is not None and (streamed.strip() or reasoned.strip()) and usage
      and usage.get('prompt_tokens', 0) > 0 and usage.get('completion_tokens', 0) > 0,
      f'first token {first and round(first, 2)}s, {usage and usage.get("completion_tokens")} completion tokens')
print('   sample:', repr((streamed.strip() or reasoned.strip())[:160]))

# 4. Held-out validation rows: does the served model behave like the dsh agent?
path = '/kaggle/working/dist/validation.jsonl.gz'
if not os.path.exists(path):
    path = f'{DATA_DIR}/dist/validation.jsonl.gz'
rows = [json.loads(line) for line in gzip.open(path, 'rt', encoding='utf-8')]

def first_expected(row):
    for i, message in enumerate(row['messages']):
        if message['role'] == 'assistant':
            calls = message.get('tool_calls') or []
            return i, (calls[0]['function']['name'] if calls else None)
    return None, None

wanted = ['dev.bugfix', 'people.ask_user', 'memory.remember', 'team.delegate_parallel']
picked = []
for category in wanted:
    for row in rows:
        ai, expected = first_expected(row)
        if row['category'] == category and expected:
            picked.append((row, ai, expected))
            break
check('held-out validation rows are available', len(picked) == len(wanted), f'{len(picked)}/{len(wanted)}')

matched = 0
for row, ai, expected in picked:
    tools = row['tools']
    try:
        resp = client.chat.completions.create(model=MODEL, messages=row['messages'][:ai], tools=tools,
                                              tool_choice='auto', temperature=0, max_tokens=1024)
    except Exception as exc:
        check(f'{row["category"]}: calls {expected}', False, str(exc)[:140])
        continue
    message = resp.choices[0].message
    call = (message.tool_calls or [None])[0]
    got = call.function.name if call else None
    args_ok = False
    if call:
        schema = next((t.get('function', t).get('parameters') for t in tools
                       if t.get('function', t).get('name') == got), None)
        try:
            validate(json.loads(call.function.arguments), schema)
            args_ok = True
        except Exception:
            args_ok = False
    matched += got == expected
    check(f'{row["category"]}: calls {expected}', got == expected and args_ok,
          f'got {got}, arguments valid={args_ok}')
check('behaves like the dsh agent (3 of 4 held-out tool choices)', matched >= 3, f'{matched}/{len(picked)}')

# 5. Tool calls also have to stream as deltas (what dsh consumes).
if picked:
    row, ai, expected = picked[0]
    body = {'model': MODEL, 'messages': row['messages'][:ai], 'tools': row['tools'], 'tool_choice': 'auto',
            'temperature': 0, 'max_tokens': 1024, 'stream': True,
            'stream_options': {'include_usage': True}}
    name = arguments = ''
    with requests.post(f'{BASE}/chat/completions', json=body, stream=True, timeout=300,
                       headers={**HEADERS, 'Accept': 'text/event-stream'}) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data:'):
                continue
            payload = line[5:].strip()
            if payload == '[DONE]':
                break
            for choice in json.loads(payload).get('choices', []):
                for call in (choice.get('delta') or {}).get('tool_calls') or []:
                    fn = call.get('function') or {}
                    name += fn.get('name') or ''
                    arguments += fn.get('arguments') or ''
    check('tool calls stream as OpenAI deltas', name == expected and arguments.lstrip().startswith('{'),
          f'name={name!r} arguments={arguments[:80]!r}')

# 6. The fine-tune, not base Qwen: the dsh persona is in the weights.
system = next((m['content'] for m in rows[0]['messages'] if m['role'] == 'system'), '')
resp = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'system', 'content': system},
                           {'role': 'user', 'content': 'In one short sentence, who are you?'}],
    temperature=0, max_tokens=512)
message = resp.choices[0].message
reasoning = getattr(message, 'reasoning_content', None) or (message.model_extra or {}).get('reasoning_content', '')
answer = ((message.content or '') + ' ' + (reasoning or '')).strip()
check('answers as decillion, not as base Qwen', 'decillion' in answer.lower(), repr(answer[:140]))

# 7. The provider to paste into Decillion (Admin → AI → Providers → Add → Custom).
gateway = {'name': 'Decillion dsh Qwen3.5-4B (Kaggle vLLM)', 'kind': 'custom', 'base_url': BASE,
           'api_key': KEY, 'enabled': True,
           'models': [{'id': MODEL, 'name': 'Decillion dsh Qwen3.5-4B',
                       'context_window': SERVE['max_model_len'], 'capabilities': ['chat'],
                       'input_price': 0, 'output_price': 0}]}
json.dump(gateway, open(f'{OUT_DIR}/decillion_gateway_provider.json', 'w'), indent=1)
print('\nprovider for Decillion (also saved to decillion_gateway_provider.json):')
print(json.dumps(gateway, indent=1))

failed = [name for name, ok in results if not ok]
print(f'\n{len(results) - len(failed)}/{len(results)} checks passed')
if failed:
    raise AssertionError('failed: ' + '; '.join(failed))
print('The fine-tuned model is served correctly. Keep this runtime (and cloudflared) alive while')
print('the provider is in use; after a disconnect rerun cells 13–14 and update the base URL.')